# Tokenizers, Models and Text Generation: Opening the LLM Black Box

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/06-transformers/02-Tokenizer-Model-Generate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommended environment: Google Colab with a T4 GPU (`Runtime → Change runtime type → T4 GPU`).

In [ ]:
!pip install -q -U transformers
# Download the reference solutions for this notebook's exercises (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/06-transformers/'
for fn in ['q_tokens.py', 'q_sampling.py', 'q_perplexity.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

In [ ]:
import time
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
device = "cuda" if torch.cuda.is_available() else "cpu"

## Tokenizer

An LLM doesn't see text; it sees **token** IDs. Most modern LLMs use BPE (Byte Pair Encoding): frequent strings are merged into a single token.

The example mixes English with a Chinese sentence ("The weather is really nice today.") on purpose, so you can see how non-English text is split.

In [ ]:
model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
print("Vocabulary size:", len(tokenizer))
ids = tokenizer.encode("Hello world! 今天天氣真好。")
ids

In [ ]:
[tokenizer.decode([i]) for i in ids]

In [ ]:
tokenizer.decode(ids)

In [ ]:
# Process several sentences at once, pad them to the same length, and return PyTorch tensors
batch = tokenizer(["Hello", "What is a Transformer?"], padding=True, return_tensors="pt")
batch

## Q
* How many tokens do a Chinese sentence and an English sentence with the same meaning need?
* How are the number `"1234567"`, emoji, and code split into tokens?

In [ ]:
# %load q_tokens.py
%run -i q_tokens.py

## Loading the Model

* `AutoModelForCausalLM`: a "causal language model", i.e. a model that predicts the next token (GPT-style)
* The T4 doesn't support bf16, so we specify `dtype=torch.float16`
* `device_map="auto"` places the model on the GPU automatically

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto")
print(model.config.num_hidden_layers, "layers,", model.config.hidden_size, "hidden size")
print(f"{model.num_parameters()/1e9:.2f}B parameters, memory: {model.get_memory_footprint()/1e9:.2f} GB")

In [ ]:
model

## Predicting the Next Token

The model outputs `logits` with shape `(batch, sequence length, vocabulary size)`: at each position, a score for every candidate "next token".

In [ ]:
inputs = tokenizer("The highest mountain in Taiwan is", return_tensors="pt").to(device)
with torch.no_grad():
    out = model(**inputs)
out.logits.shape

In [ ]:
probs = out.logits[0, -1].float().softmax(-1)
top = torch.topk(probs, 10)
for p, i in zip(top.values, top.indices):
    print(f"{tokenizer.decode(i)!r:12} {p.item():.3f}")

### Writing Our Own Generation Loop (Greedy Decoding)

Pick the single most likely token, append it, then predict the next one.

In [ ]:
@torch.no_grad()
def greedy(prompt, max_new_tokens=30):
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    for _ in range(max_new_tokens):
        logits = model(ids).logits[:, -1]
        next_id = logits.argmax(-1, keepdim=True)
        ids = torch.cat([ids, next_id], dim=1)
        if next_id.item() == tokenizer.eos_token_id:
            break
    return tokenizer.decode(ids[0])
t0 = time.perf_counter()
print(greedy("The highest mountain in Taiwan is", 50))
print(f"{time.perf_counter()-t0:.1f}s")

Hmm, the model doesn't just give a clean answer: it keeps rambling, adding more facts or even inventing new questions of its own. That's because Qwen3-0.6B is a **chat model**, trained on text wrapped in a conversation format.
Given raw text, it simply "keeps writing" whatever is most likely to come next. The **chat template** introduced below fixes this.

### KV Cache

The loop above recomputes the whole sequence at every step, which is wasteful. In attention, the keys and values of earlier positions never change, so we can store and reuse them. This is the **KV cache**.

In [ ]:
@torch.no_grad()
def greedy_kv(prompt, max_new_tokens=30):
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    out = model(ids, use_cache=True)
    generated = []
    for _ in range(max_new_tokens):
        next_id = out.logits[:, -1].argmax(-1, keepdim=True)
        generated.append(next_id.item())
        if next_id.item() == tokenizer.eos_token_id:
            break
        out = model(next_id, past_key_values=out.past_key_values, use_cache=True)  # only feed the new token
    return tokenizer.decode(ids[0].tolist() + generated)
t0 = time.perf_counter()
print(greedy_kv("The highest mountain in Taiwan is", 50))
print(f"{time.perf_counter()-t0:.1f}s")

In [ ]:
type(model(**inputs, use_cache=True).past_key_values)

The prompt above is short, so recomputing it on the GPU is fast and the difference is hard to see. Let's compare with a longer prompt (about 1000 tokens):

In [ ]:
long_prompt = "Yushan is the highest mountain in Taiwan, at 3,952 meters. " * 60 + "The highest mountain in Taiwan is"
print("prompt tokens:", len(tokenizer(long_prompt).input_ids))
for f in [greedy, greedy_kv]:
    t0 = time.perf_counter()
    f(long_prompt, 50)
    print(f"{f.__name__:10s} {time.perf_counter() - t0:.1f}s")

### `model.generate`

In practice, `generate` already handles the KV cache, various sampling methods, stopping criteria, and more.

In [ ]:
inputs = tokenizer("The highest mountain in Taiwan is", return_tensors="pt").to(device)
out = model.generate(**inputs, max_new_tokens=50, do_sample=False)
tokenizer.decode(out[0])

In [ ]:
# Sampling
out = model.generate(**inputs, max_new_tokens=50, do_sample=True, temperature=0.7, top_p=0.8, top_k=20,
                     num_return_sequences=3)
for o in out:
    print(tokenizer.decode(o, skip_special_tokens=True))
    print("-----")

## Q
Implement your own sampling generation loop (with KV cache) that supports `temperature` and `top_p` (nucleus sampling):
* top-p: sort the probabilities from largest to smallest, keep the smallest set whose cumulative probability reaches `p`, and sample from that set

In [ ]:
# %load q_sampling.py
%run -i q_sampling.py

## Chat Template

Chat models (instruct models) are trained on conversations wrapped in a specific format. Every model has its own format, and `apply_chat_template` uses the template that ships with the model.

In [ ]:
messages = [
    {"role": "system", "content": "You are a concise teaching assistant."},
    {"role": "user", "content": "What is a Transformer?"},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
print(text)

In [ ]:
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False,
                                       return_dict=True, return_tensors="pt").to(device)
out = model.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.7, top_p=0.8, top_k=20)
print(tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True))

## Thinking Mode (Reasoning)

"Reasoning models" like Qwen3 first write out their thought process inside `<think> ... </think>`, then answer. The longer they think, the better they usually do on hard problems (math, code).

In [ ]:
from transformers import TextStreamer
messages = [{"role": "user", "content": "A cage holds chickens and rabbits. There are 35 heads and 94 legs in total. How many chickens and how many rabbits are there?"}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=True,
                                       return_dict=True, return_tensors="pt").to(device)
streamer = TextStreamer(tokenizer, skip_prompt=True)   # print the text as it is generated
out = model.generate(**inputs, max_new_tokens=2048, do_sample=True, temperature=0.6, top_p=0.95, top_k=20,
                     streamer=streamer)

In [ ]:
# Separate the thinking from the final answer
gen = out[0, inputs.input_ids.shape[1]:].tolist()
think_end = tokenizer.convert_tokens_to_ids("</think>")
k = len(gen) - gen[::-1].index(think_end) if think_end in gen else 0
print("Thought for", k, "tokens")
print("Answer:", tokenizer.decode(gen[k:], skip_special_tokens=True))

## Q: Perplexity

A language model's loss on a piece of text (the average negative log-likelihood per token), exponentiated, is the perplexity. Lower means the model finds the text more "natural".

Hint: `model(input_ids, labels=input_ids).loss`; the model shifts the labels by one position automatically to compute the cross entropy.

Compare the perplexity of these sentences. The Chinese ones are intentional: a natural sentence ("The weather is nice today, let's go for a walk in the park."), the same sentence with two words swapped, and a scrambled version. The English ones follow the same pattern.
* `"今天天氣很好，我們去公園散步吧。"`
* `"今天公園很好，我們去天氣散步吧。"`
* `"天今很好氣，我公散們去步園吧。"`
* `"The weather is nice today, let's go for a walk in the park."`
* `"The park is nice today, let's go for a walk in the weather."`
* `"weather The nice is, today walk go let's a for park the in."`

In [ ]:
# %load q_perplexity.py
%run -i q_perplexity.py